# 🌧️ Day 24 · Network: where does a raindrop go?

**#30DayMapChallenge Uzbekistan 2026** · beginner tutorial, no experience needed

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NodiraTillayeva/30DayMapChallengeUzbekistan/blob/main/2026/Day%2024%20Network/tutorial/day24_where_does_a_raindrop_go.ipynb)

Rivers are a **network**. Every stream flows into a bigger one, and that one into a bigger one, all the way to the end. If you know which piece flows into which, you can answer questions like:
- a raindrop falls on Chimgan: where does it end up, and how far does it travel?
- which mountains does the water in the Chirchik come from?
- how much of the water in our big rivers comes from our neighbours?

In this notebook you will:
1. download **HydroRIVERS**, a map of every river on Earth,
2. turn the rivers of Central Asia into a **network** with **networkx**,
3. follow raindrops **downstream** from six places,
4. go **upstream** to draw the whole basin of a river,
5. find out which countries the water of the Amu Darya and the Syr Darya comes from,
6. save a poster, and drop your own raindrop.

Run the cells from top to bottom: click a cell, then press **Shift + Enter**.

## Step 0 · Get ready

**HydroRIVERS** is made by the HydroSHEDS project from satellite elevation data: water always flows downhill, so the computer can work out where every river goes. It's free, but you download it yourself from the HydroSHEDS website: by downloading it, you accept their [license](https://www.hydrosheds.org/products/hydrorivers).

The file for all of Asia is about 90 MB. `!wget` downloads it and `!unzip` unpacks it (the `!` runs a command outside Python). Then we read only the box around Central Asia, and only rivers that collect water from at least **100 km²**.

We also load the country borders of Central Asia from **Natural Earth**.

In [ ]:
!wget -q -nc https://data.hydrosheds.org/file/HydroRIVERS/HydroRIVERS_v10_as_shp.zip
!unzip -q -n HydroRIVERS_v10_as_shp.zip

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt
from shapely.geometry import Point

rivers = gpd.read_file("HydroRIVERS_v10_as_shp/HydroRIVERS_v10_as.shp",
                       bbox=(55, 34, 80, 47), where="UPLAND_SKM >= 100")
rivers = rivers.set_index("HYRIV_ID")

countries = gpd.read_file("https://raw.githubusercontent.com/NodiraTillayeva/30DayMapChallengeUzbekistan/main/2026/Day%2024%20Network/tutorial/data/central_asia_countries.geojson")
uzbekistan = countries[countries["country"] == "Uzbekistan"]

METRES = "EPSG:32642"     # UTM zone 42N: for distances in metres

BACK, LAND, LINE, WATER = "#0e1a24", "#1b2a36", "#3c5263", "#4fb3e8"
def base_map(ax, xlim=(55, 80), ylim=(34, 47)):
    ax.set_facecolor(BACK)
    countries.plot(ax=ax, color=LAND, edgecolor=LINE, linewidth=0.6)
    uzbekistan.plot(ax=ax, color="#22394a", edgecolor="#7d97aa", linewidth=0.8)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_axis_off()

print(len(rivers), "river pieces. Ready! ✅")

## Step 1 · Look at the rivers

Each row is one **piece** of river, a few km long. The most useful columns:
- `NEXT_DOWN`: the ID of the piece this one flows into (0 = the river ends here),
- `LENGTH_KM`: its length,
- `UPLAND_SKM`: how much land drains into it, in km²,
- `DIS_AV_CMS`: how much water flows in it on average, in m³ per second.

Here are the 5 pieces with the most water:

In [ ]:
rivers[["NEXT_DOWN", "LENGTH_KM", "UPLAND_SKM", "DIS_AV_CMS"]].nlargest(5, "DIS_AV_CMS")

In [ ]:
width = (np.log10(rivers["DIS_AV_CMS"] + 1) * 0.8).clip(0.1, 3)

fig, ax = plt.subplots(figsize=(14, 8), facecolor=BACK)
base_map(ax)
rivers.plot(ax=ax, color=WATER, linewidth=width)
plt.show()

The thick lines are the big rivers: the **Amu Darya** coming from the Pamirs in the south-east, and the **Syr Darya** coming from the Tian Shan through the Fergana valley. Most of the thin lines are in the **mountains**: that's where the water is born.

## Step 2 · Rivers as a network

A **network** (or **graph**) has **nodes** and **edges**. Here, every river piece is a node, and every "flows into" is an edge with a direction: an arrow from a piece to its `NEXT_DOWN`. **networkx** is a Python library for networks.

In [ ]:
G = nx.DiGraph()                     # Di = directed: the arrows have a direction
G.add_nodes_from(rivers.index)
G.add_edges_from((piece, down) for piece, down in rivers["NEXT_DOWN"].items() if down in rivers.index)

print(G.number_of_nodes(), "nodes (river pieces)")
print(G.number_of_edges(), "edges (flows into)")
print(sum(1 for n in G if G.out_degree(n) == 0), "ends (pieces that flow into nothing)")

Some "ends" are real (a river runs into a lake or the desert), and some are only at the edge of our box.

Each piece flows into **at most one** piece: water can't split. But **many** pieces can flow into one: that's where tributaries join. A network like this is called a **tree**.

`G.successors(piece)` gives the piece downstream, `G.predecessors(piece)` the pieces upstream.

## Step 3 · A raindrop on Chimgan

We drop a raindrop on Chimgan, find the nearest river piece, and follow the arrows until the river ends. In a river network there's only one way down, so `nx.dfs_preorder_nodes` (walk along the arrows) gives exactly the path.

In [ ]:
rivers_m = rivers.to_crs(METRES)

def nearest_piece(lat, lng, min_upland=0):
    here = gpd.GeoSeries([Point(lng, lat)], crs=4326).to_crs(METRES).iloc[0]
    big_enough = rivers_m[rivers_m["UPLAND_SKM"] >= min_upland]
    return big_enough.distance(here).idxmin()

def downstream(lat, lng):
    start = nearest_piece(lat, lng)
    return list(nx.dfs_preorder_nodes(G, start))

path = downstream(41.55, 70.05)       # Chimgan
km = rivers.loc[path, "LENGTH_KM"].sum()
print("The raindrop travels", round(km), "km through", len(path), "river pieces")
print("At about 1 m/s, that takes about", round(km * 1000 / 1 / 86400), "days")

fig, ax = plt.subplots(figsize=(14, 8), facecolor=BACK)
base_map(ax)
rivers.plot(ax=ax, color=WATER, linewidth=width * 0.5, alpha=0.4)
rivers.loc[path].plot(ax=ax, color="#ffd166", linewidth=2.5)
ax.plot(70.05, 41.55, "o", color="#ffd166", markersize=8)
plt.show()

From Chimgan, the raindrop goes into the **Chirchik**, past Tashkent into the **Syr Darya**, through Kazakhstan, and to the end of the Syr Darya: almost **1,800 km**, about three weeks.

## Step 4 · A raindrop race

Six raindrops, six places. Where does each one end, and how far does it travel? We name the end by its river: the piece where the Amu Darya and the Syr Darya end is in their `MAIN_RIV` column.

In [ ]:
amu = nearest_piece(37.22, 67.28, min_upland=100_000)      # the Amu Darya at Termez
syr = nearest_piece(40.22, 69.27, min_upland=50_000)       # the Syr Darya at Bekabad
RIVER = {rivers.at[amu, "MAIN_RIV"]: "Amu Darya", rivers.at[syr, "MAIN_RIV"]: "Syr Darya"}

drops = {
    "Chimgan":              (41.55, 70.05),
    "Andijan":              (40.78, 72.35),
    "Zarafshan glacier":    (39.50, 70.57),
    "Shahrisabz":           (38.95, 66.90),
    "Baysun mountains":     (38.30, 67.60),
    "Nurata mountains":     (40.60, 66.20),
}

rows, paths = [], {}
for name, (lat, lng) in drops.items():
    p = downstream(lat, lng)
    paths[name] = p
    rows.append({"raindrop": name,
                 "km": round(rivers.loc[p, "LENGTH_KM"].sum()),
                 "ends in": RIVER.get(rivers.at[p[-1], "MAIN_RIV"], "the desert 🏜️")})
pd.DataFrame(rows)

In [ ]:
COLOURS = ["#ffd166", "#ef476f", "#06d6a0", "#f78c6b", "#c77dff", "#ffffff"]

fig, ax = plt.subplots(figsize=(14, 8), facecolor=BACK)
base_map(ax)
rivers.plot(ax=ax, color=WATER, linewidth=width * 0.5, alpha=0.3)
for (name, p), colour in zip(paths.items(), COLOURS):
    rivers.loc[p].plot(ax=ax, color=colour, linewidth=2)
    lat, lng = drops[name]
    ax.plot(lng, lat, "o", color=colour, markersize=7)
    ax.text(lng + 0.2, lat + 0.15, name, color=colour, fontsize=9)
plt.show()

Almost every raindrop ends in one of the two big rivers. Even from **Shahrisabz** and the **Zarafshan glacier**, the map sends the water to the Amu Darya. But the raindrop from the **Nurata mountains** never reaches a big river: its stream runs into the desert after about 170 km and disappears.

⚠️ This is the **natural** network: where water would flow **downhill**. People have changed it. Today, almost all the water of the Zarafshan and the Kashkadarya is taken into canals and fields, and they don't reach the Amu Darya any more.

## Step 5 · Go upstream: the basin of the Chirchik

Now the other way. `nx.ancestors` gives **every** piece that flows into a piece, however far away. All the land that drains into one place is called its **basin**.

Where does the water of the Chirchik at Tashkent come from?

In [ ]:
chirchik = nearest_piece(41.20, 69.15, min_upland=5_000)    # the Chirchik near Tashkent
basin = list(nx.ancestors(G, chirchik)) + [chirchik]

print(len(basin), "river pieces flow into the Chirchik at Tashkent")
print("Land that drains here:", round(rivers.at[chirchik, "UPLAND_SKM"]), "km²")

fig, ax = plt.subplots(figsize=(10, 8), facecolor=BACK)
base_map(ax, xlim=(68.5, 72), ylim=(40.5, 42.8))
rivers.plot(ax=ax, color=WATER, linewidth=width * 0.6, alpha=0.3)
rivers.loc[basin].plot(ax=ax, color="#ffd166", linewidth=width.loc[basin] * 1.2)
plt.show()

The Chirchik collects water from about **12,500 km²** of mountains: the Pskem and the Ugam in the north, the Chatkal in the east, which starts in **Kyrgyzstan**. Charvak (Day 23) is where they meet.

## Step 6 · Whose water is it?

The water that flows in a river piece = the water that came from upstream + the water from its own little piece of land. So for each piece:

**local water = its own flow − the flow of the pieces that flow into it**

Then we find which country each piece is in and add up the local water by country.

In [ ]:
flow = rivers["DIS_AV_CMS"]
from_upstream = pd.Series({p: sum(flow[q] for q in G.predecessors(p)) for p in rivers.index})
rivers["local"] = (flow - from_upstream).clip(lower=0)

middle = rivers_m.copy()
middle["geometry"] = rivers_m.geometry.interpolate(0.5, normalized=True)    # the middle point of each piece
middle = middle.to_crs(countries.crs)
rivers["country"] = gpd.sjoin(middle, countries, how="left", predicate="within")["country"]

def whose_water(piece):
    b = rivers.loc[list(nx.ancestors(G, piece)) + [piece]]
    return (b.groupby("country")["local"].sum() / b["local"].sum() * 100).round(1).sort_values(ascending=False)

table = pd.DataFrame({"Amu Darya at Termez": whose_water(amu),
                      "Syr Darya at Bekabad": whose_water(syr)}).fillna(0)
table

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 4))
for ax, col in zip(axs, table.columns):
    t = table[col][table[col] > 0.5].sort_values()
    ax.barh(t.index, t.values, color=["#ffd166" if c == "Uzbekistan" else "#4fb3e8" for c in t.index])
    ax.set_title(col)
    ax.set_xlabel("% of the water")
plt.tight_layout()
plt.show()

When the Amu Darya reaches **Termez**, about two thirds of its water comes from **Tajikistan** and more than a quarter from **Afghanistan**. When the Syr Darya enters Uzbekistan near **Bekabad**, about **nine tenths** of its water comes from **Kyrgyzstan**. The rivers that feed Uzbekistan are born in the mountains of its neighbours. That's why water is something the countries of Central Asia have to **share**.

## Step 7 · Save your poster

All the rivers, and the six raindrop paths. No title, no labels. It saves **where_does_a_raindrop_go.png**.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9), facecolor=BACK)
ax.set_facecolor(BACK)
countries.plot(ax=ax, color=BACK, edgecolor="#2a3d4c", linewidth=0.5)
rivers.plot(ax=ax, color="#2f6f99", linewidth=width * 0.6)
for (name, p), colour in zip(paths.items(), COLOURS):
    rivers.loc[p].plot(ax=ax, color=colour, linewidth=1.8)
    lat, lng = drops[name]
    ax.plot(lng, lat, "o", color=colour, markersize=5)
ax.set_xlim(55, 80); ax.set_ylim(34, 47)
ax.axis("off")
plt.savefig("where_does_a_raindrop_go.png", dpi=200, bbox_inches="tight", facecolor=BACK)

try:
    from google.colab import files
    files.download("where_does_a_raindrop_go.png")
except ImportError:
    print("Saved where_does_a_raindrop_go.png")

## Step 8 · Your own raindrop

Drop a raindrop on your town, your school or your favourite mountain. On [openstreetmap.org](https://www.openstreetmap.org), right-click a place and choose **Show address** to see its coordinates. Change `lat, lng` and run the cell.

In [ ]:
lat, lng = 39.65, 66.96          # Samarkand

p = downstream(lat, lng)
print("Nearest river piece is", round(rivers_m.loc[p[0]].geometry.distance(
      gpd.GeoSeries([Point(lng, lat)], crs=4326).to_crs(METRES).iloc[0]) / 1000, 1), "km away")
print("The raindrop travels", round(rivers.loc[p, "LENGTH_KM"].sum()), "km and ends in",
      RIVER.get(rivers.at[p[-1], "MAIN_RIV"], "the desert 🏜️"))

fig, ax = plt.subplots(figsize=(14, 8), facecolor=BACK)
base_map(ax)
rivers.plot(ax=ax, color=WATER, linewidth=width * 0.5, alpha=0.4)
rivers.loc[p].plot(ax=ax, color="#ffd166", linewidth=2.5)
ax.plot(lng, lat, "o", color="#ffd166", markersize=8)
plt.show()

## 🎉 Done!

You built a river network, followed raindrops downstream, went upstream to draw a basin, and found out where the water of Uzbekistan's big rivers comes from.

Share your map with **#30DayMapChallenge** and **#30DayMapChallengeUzbekistan**, and please credit:
> Rivers: HydroRIVERS (Lehner & Grill 2013), www.hydrosheds.org. Borders: Natural Earth.